In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import torch
import networkx as nx

In [ ]:
print(torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0))

True 1 NVIDIA GeForce RTX 2080 Ti


In [ ]:
#implementing another deduplication, as tfidf one didn't detect template-basede duplicates that were missing some paragraphs/were slightl paraphrased
# Only compares documents that share the same `project` value.

# Grouping logic uses complete-linkage (cliques) instead of union-find (previously tested version, also used in tf-idf):
# union-find merges transitively (A~B and B~C -> A,B,C grouped together even if A and C aren't actually similar),
# which caused chaining - tested on this dataset, one project chained 177 documents into a  single "duplicate" group with a minimum pairwise similarity of just 0.156.
# Requiring every pair in a group to clear the threshold (a clique) fixes this.

def run_semantic_deduplication(dataset, project_col="project", threshold=0.90, model_name="BAAI/bge-m3", output_filename="candidate_semantic_duplicate_groups.xlsx"):

    df_documents = dataset.copy()
    df_documents["dedup_text"] = df_documents["text"].fillna("").astype(str).str.strip()
    df_documents = df_documents[df_documents["dedup_text"].str.len() > 0].reset_index(drop=True).copy()
    df_documents["word_count"] = df_documents["dedup_text"].str.split().str.len()

    print("Loading embedding model")
    model = SentenceTransformer(model_name, device="cuda:0")
    # bge-m3 defaults to max_seq_length=8192, which covers SCOT-BESS's full length distirbutions
    print(f"Model max_seq_length: {model.max_seq_length}")
    model.half()

    embedding_store = {}

    graph = nx.Graph()

    project_blocks = df_documents.groupby(project_col).indices
    n_projects = len(project_blocks)
    n_singleton_blocks = sum(1 for idxs in project_blocks.values() if len(idxs) < 2)
    print(f"Projects: {n_projects} total, {n_singleton_blocks} have only one document (skipped, nothing to compare within them)")

    for project, idx_array in project_blocks.items():
        if len(idx_array) < 2:
            continue

        texts = df_documents.loc[idx_array, "dedup_text"].tolist()
        embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True, batch_size=8)

        for idx, emb in zip(idx_array, embeddings):
            embedding_store[idx] = emb

        similarity_matrix = embeddings @ embeddings.T
        upper_triangle = np.triu(similarity_matrix, k=1)
        rows, cols = np.where(upper_triangle >= threshold)

        for r, c in zip(rows, cols):
            graph.add_edge(int(idx_array[r]), int(idx_array[c]), weight=float(similarity_matrix[r, c]))

    # complete-linkage grouping

    duplicate_groups = []

    for component in nx.connected_components(graph):
        subgraph = graph.subgraph(component)
        cliques = list(nx.find_cliques(subgraph))
        cliques.sort(key=len, reverse=True)

        assigned = set()
        for clique in cliques:
            remaining = [n for n in clique if n not in assigned]
            if len(remaining) > 1:
                duplicate_groups.append(sorted(remaining))
                assigned.update(remaining)

    duplicate_groups = sorted(duplicate_groups, key=lambda group: min(group))

    group_preview = []

    for group_id, group in enumerate(duplicate_groups, start=1):
        longest_index = max(group, key=lambda idx: df_documents.loc[idx, "word_count"])
        keep_embedding = embedding_store[longest_index]

        group_embeddings = np.array([embedding_store[idx] for idx in group])
        group_sim = group_embeddings @ group_embeddings.T
        np.fill_diagonal(group_sim, 1.0)
        min_intragroup_similarity = float(group_sim.min())

        for idx in group:
            similarity_to_keep = (
                1.0 if idx == longest_index
                else float(embedding_store[idx] @ keep_embedding)
            )
            group_preview.append({
                "group_id": group_id,
                "document_id": int(df_documents.loc[idx, "document_id"]),
                "project": df_documents.loc[idx, project_col],
                "filename": df_documents.loc[idx, "filename"],
                "source": df_documents.loc[idx, "source"],
                "word_count": int(df_documents.loc[idx, "word_count"]),
                "similarity_to_suggested_keep": round(similarity_to_keep, 4),
                "min_intragroup_similarity": round(min_intragroup_similarity, 4),
                "suggested_keep": idx == longest_index,
                "text": df_documents.loc[idx, "dedup_text"]})

    df_duplicate_groups = pd.DataFrame(group_preview)
    #xlsx for easier inspection
    df_duplicate_groups.to_excel(output_filename, index=False)

    print(f"\nModel: {model_name}")
    print(f"Threshold: {threshold}")
    print(f"Candidate semantic duplicate groups (cliques): {len(duplicate_groups)}")
    print(f"Supposed duplicates after keeping one per group: {sum(len(group) - 1 for group in duplicate_groups)}")

    return df_duplicate_groups

In [ ]:
dataset = pd.read_csv("SCOTBESS_DATASET_DEDUPLICATED_80.csv")
df_semantic_duplicate_groups_92 = run_semantic_deduplication(dataset, threshold=0.92, output_filename="candidate_semantic_duplicate_groups_92_cliques.xlsx",)

Loading embedding model


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model max_seq_length: 8192
Projects: 91 total, 19 have only one document (skipped, nothing to compare within them)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Model: BAAI/bge-m3
Threshold: 0.92
Candidate semantic duplicate groups (cliques): 160
Supposed duplicates after keeping one per group: 924


In [ ]:
document_ids_to_remove = df_semantic_duplicate_groups_92.loc[~df_semantic_duplicate_groups_92["suggested_keep"], "document_id"].drop_duplicates()
df_dataset_deduplicated_semantic_92 = dataset.loc[~dataset["document_id"].isin(document_ids_to_remove)].reset_index(drop=True).copy()

df_dataset_deduplicated_semantic_92.to_csv("SCOTBESS_DATASET_DEDUPLICATED_SEMANTIC_92.csv", index=False, encoding="utf-8")
df_dataset_deduplicated_semantic_92.to_excel("SCOTBESS_DATASET_DEDUPLICATED_SEMANTIC_92.xlsx", index=False)